In [ ]:
import os
from collections import defaultdict

import kagglehub
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.image import imread
import pandas as pd
import seaborn as sns
from PIL import Image
from tqdm import tqdm

os.chdir('..')

In [ ]:
# Get the dataset path
path = kagglehub.dataset_download("gpiosenka/sports-classification")

print(f"Dataset location: {path}\n")

### Core Statistics

In [ ]:
# Load and analyze CSV
df = pd.read_csv(os.path.join(path, "sports.csv"))
print(df.head())
# df.head().to_csv("output/head.csv", index=False)
print("\n" + "="*60)

# Analyze folder structure and file types from actual filesystem
print(f"\nContents of directory:")
print(os.listdir(path))

print("\n" + "="*60)
print("FOLDER STRUCTURE:\n")

folder_counts = defaultdict(int)
file_type_counts = defaultdict(int)

for root, dirs, files in os.walk(path):
    for file in files:
        # Get the relative path from path
        rel_path = os.path.relpath(os.path.join(root, file), path)
        top_folder = rel_path.split(os.sep)[0]
        folder_counts[top_folder] += 1
        
        # Get file extension
        _, ext = os.path.splitext(file)
        if ext:  # Only count if there's an extension
            file_type_counts[ext] += 1

for folder in sorted(folder_counts.keys()):
    print(f"  {folder}/: {folder_counts[folder]} files")

print("\nFILE TYPES:")
for ext in sorted(file_type_counts.keys()):
    print(f"  {ext}: {file_type_counts[ext]} files")

# Basic statistics
print(f"TOTAL IMAGES: {len(df)}")
print(f"TOTAL CLASSES: {df['labels'].nunique()}")

# Visualize class distribution
counts = df["labels"].value_counts()
plt.figure(figsize=(10, 15))
sns.barplot(y=counts.index, x=counts.values, hue=counts.index, palette='inferno', legend=False)
plt.title('Amount of images per label')
plt.ylabel('Labels')
plt.xlabel('Count')
for i, v in enumerate(counts.values):
    plt.text(v + 0.5, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig("output/dataset_distribution_100.png", dpi=150)
print(f"Figure saved to: {os.getcwd()}/output/dataset_distribution_100.png")
plt.show()

### Train/test/valid ratios

In [ ]:
# sports = ["football", "air hockey", "basketball"]  # replace with your labels
sports = df['labels'].unique().tolist()
counts = (
    df[df['labels'].isin(sports)]
    .groupby(['labels', 'data set'])
    .size()
    .unstack(fill_value=0)
)

ratios = counts.div(counts.sum(axis=1), axis=0)
print(ratios.round(3))

mean_ratios = ratios.mean(axis=0)
print(mean_ratios.round(3))

### Data Quality & Characteristics

In [ ]:
# Calculate image sizes
print("\nCalculating image sizes...")
 
def get_image_size(filepath):
    try:
        with Image.open(filepath) as img:
            width, height = img.size
            return [width, height]  # Store as [width, height]
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return None
 
df["image_size"] = [
    get_image_size(os.path.join(path, fp)) 
    for fp in tqdm(df["filepaths"], desc="Files")
]
 
# Display statistics
print(f"\nImage sizes added!")
 
# Extract widths and heights for statistics
widths = df["image_size"].apply(lambda x: x[0] if x else None)
heights = df["image_size"].apply(lambda x: x[1] if x else None)
pixel_counts = df["image_size"].apply(lambda x: x[0] * x[1] if x else None)
 
print(f"Width  - Min: {widths.min()}, Max: {widths.max()}, Mean: {widths.mean():.0f}")
print(f"Height - Min: {heights.min()}, Max: {heights.max()}, Mean: {heights.mean():.0f}")
print(f"Resolution - Min: {pixel_counts.min()}, Max: {pixel_counts.max()}, Mean: {pixel_counts.mean():.0f}")
print(f"Missing sizes: {df['image_size'].isna().sum()} images")

### Class-Specific Insights

In [ ]:
grouped = df.groupby('labels').size().reset_index(name='count')
max_idx = grouped['count'].idxmax()
min_idx = grouped['count'].idxmin()

max_row = grouped.loc[max_idx]
name_max = max_row['labels']
count_max = max_row['count']

min_row = grouped.loc[min_idx]
name_min = min_row['labels']
count_min = min_row['count']

print(f"The {name_max} class ({count_max} images) is {count_max/count_min:.2f} times bigger than the {name_min} class ({count_min} images).")

### Show example images

Since amount of images is different for each sport, we can just use the first image for display

In [ ]:
sports = ["air hockey", "basketball", "tennis", "swimming",
          "football", "volleyball", "baseball", "bungee jumping"]
split = "test"

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for sport, ax in zip(sports, axes.flat):
    img_path = os.path.join(path, f"{split}/{sport}/1.jpg")
    img = imread(img_path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(sport.title())

plt.tight_layout()
plt.savefig("output/sample_images.png", dpi=150)
plt.show()

### Class Distribution(top and bottom 10)

In [ ]:
# Top 10 and bottom 10 classes by image count
top10 = counts.head(10)
bottom10 = counts.tail(10).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.barplot(y=top10.index, x=top10.values, hue=top10.index,
            palette='inferno', legend=False, ax=axes[0])
axes[0].set_title('Top 10 classes by image count')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Labels')
for i, v in enumerate(top10.values):
    axes[0].text(v + 0.5, i, str(v), va='center', fontsize=9)

sns.barplot(y=bottom10.index, x=bottom10.values, hue=bottom10.index,
            palette='viridis', legend=False, ax=axes[1])
axes[1].set_title('Bottom 10 classes by image count')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('')
for i, v in enumerate(bottom10.values):
    axes[1].text(v + 0.5, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.savefig("output/top_bottom_10_classes.png", dpi=150)
print(f"Figure saved to: {os.getcwd()}/output/top_bottom_10_classes.png")
plt.show()

### RGB mean values for given classes

In [ ]:
# sports = ["football", "air hockey", "basketball", "tennis"]
sports = ["football", "air hockey", "basketball"]
splits = ["train", "valid", "test"]

results = {}

for sport in sports:
    all_means = []

    for split in splits:
        folder = os.path.join(path, split, sport)
        if not os.path.isdir(folder):
            continue

        for filename in os.listdir(folder):
            if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            # convert("RGB") drops alpha and expands grayscale to 3 channels
            img = np.asarray(Image.open(os.path.join(folder, filename)).convert("RGB"))
            all_means.append(img.reshape(-1, 3).mean(axis=0))  # values in 0-255

    all_means = np.array(all_means)
    results[sport] = {
        "mean": all_means.mean(axis=0),
        "std": all_means.std(axis=0),
        "n_images": len(all_means),
    }

for sport, stats in results.items():
    r, g, b = stats["mean"]
    sr, sg, sb = stats["std"]
    print(f"{sport:>16} (n={stats['n_images']:>5}): "
          f"R={r:.1f}±{sr:.1f}  G={g:.1f}±{sg:.1f}  B={b:.1f}±{sb:.1f}")
channels = ["R", "G", "B"]
bar_colors = ["red", "green", "blue"]

n = len(sports)
ncols = min(n, 4)
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
flat_axes = axes.flat

for sport, ax in zip(sports, flat_axes):
    mean = results[sport]["mean"]
    std = results[sport]["std"]

    ax.bar(channels, mean, yerr=std, color=bar_colors, capsize=4)
    ax.set_title(sport.title())
    ax.set_ylim(0, 255)  # imread returns values in [0, 1]
    ax.set_ylabel("Mean intensity")

# Hide any leftover axes when the grid has more slots than sports
for ax in list(flat_axes)[n:]:
    ax.set_visible(False)

plt.suptitle("Mean RGB color per sport (all splits)")
plt.tight_layout(rect=[0, 0, 1, 0.96])  # leave room for the suptitle
# plt.savefig("output/mean_color_bars.png", dpi=150)
plt.show()

### Color calculations

In [ ]:
from sklearn.cluster import KMeans

sports = ["football", "air hockey", "basketball"]
splits = ["train", "valid", "test"]
exts = (".jpg", ".jpeg", ".png")

PIXELS_PER_IMAGE = 300   # subsample for hue histogram and palette
N_COLORS = 5             # palette size
MIN_SAT = 0.15           # near-grey pixels have meaningless hue, so skip them
rng = np.random.default_rng(0)


def image_features(filepath):
    rgb_img = Image.open(filepath).convert("RGB")
    rgb = np.asarray(rgb_img).reshape(-1, 3)
    hsv = np.asarray(rgb_img.convert("HSV")).reshape(-1, 3).astype(np.float32)

    h = hsv[:, 0] / 255 * 2 * np.pi   # hue in radians
    s = hsv[:, 1] / 255
    v = hsv[:, 2] / 255

    # Hue is circular, so average unit vectors instead of raw angles.
    # hue_conc: 0 = mixed hues, 1 = one dominant hue
    c, sn = np.cos(h).mean(), np.sin(h).mean()
    feats = np.array([
        np.arctan2(sn, c) % (2 * np.pi),  # 0: hue_mean
        np.hypot(c, sn),                  # 1: hue_conc
        s.mean(), s.std(),                # 2, 3: saturation mean/std
        v.mean(), v.std(),                # 4, 5: value mean/std
    ])

    # Subsample pixels for the histogram and palette
    idx = rng.choice(len(rgb), size=min(PIXELS_PER_IMAGE, len(rgb)), replace=False)
    hues = h[idx][s[idx] >= MIN_SAT]
    return feats, hues, rgb[idx]


# Collect per-image features and pixel samples for each sport
feats = {sp: [] for sp in sports}
hues = {sp: [] for sp in sports}
palette_px = {sp: [] for sp in sports}

for sport in sports:
    for split in splits:
        folder = os.path.join(path, split, sport)
        if not os.path.isdir(folder):
            continue
        for filename in sorted(os.listdir(folder)):
            if not filename.lower().endswith(exts):
                continue
            f, h, px = image_features(os.path.join(folder, filename))
            feats[sport].append(f)
            hues[sport].append(h)
            palette_px[sport].append(px)

feats = {sp: np.array(v) for sp, v in feats.items()}

# Summary (medians are more robust than means for skewed per-image stats)
for sport in sports:
    f = feats[sport]
    print(f"{sport:>16} (n={len(f):>4}): "
          f"sat={np.median(f[:, 2]):.2f}  val={np.median(f[:, 4]):.2f}  "
          f"hue_conc={np.median(f[:, 1]):.2f}")

# --- Plot 1: per-image distributions as box plots ---
# For violin plots, swap ax.boxplot(data) for ax.violinplot(data, showmedians=True)
metrics = [(2, "Mean saturation"), (4, "Mean brightness (value)"), (1, "Hue concentration")]
fig, axes = plt.subplots(1, 3, figsize=(5 * 3, 4.5))
for ax, (col, label) in zip(axes, metrics):
    data = [feats[sp][:, col] for sp in sports]
    # ax.boxplot(data)
    ax.violinplot(data, showmedians=True)
    ax.set_xticks(range(1, len(sports) + 1))
    ax.set_xticklabels([sp.title() for sp in sports], rotation=20)
    ax.set_title(label)
    ax.set_ylabel("Per-image value")
plt.suptitle("Per-image HSV statistics by sport")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# --- Plot 2: hue histograms on a polar axis ---
n = len(sports)
BINS = 36
edges = np.linspace(0, 2 * np.pi, BINS + 1)
centers = edges[:-1] + np.pi / BINS
width = 2 * np.pi / BINS
hue_colors = plt.cm.hsv(centers / (2 * np.pi))

fig, axes = plt.subplots(1, n, subplot_kw={"projection": "polar"},
                         figsize=(4 * n, 4.5), squeeze=False)
for ax, sport in zip(axes.flat, sports):
    h_all = np.concatenate(hues[sport])
    counts, _ = np.histogram(h_all, bins=edges)
    ax.bar(centers, counts / max(counts.sum(), 1), width=width,
           color=hue_colors, edgecolor="none")
    ax.set_title(sport.title(), pad=15)
    ax.set_yticklabels([])
plt.suptitle("Hue distribution of colorful pixels (proportion)")
plt.tight_layout()
plt.show()